# Module 02 — Supervised Learning
## 02-07: k-Nearest Neighbors

**Objective:** Implement k-NN from scratch with multiple distance metrics,
weighted voting, and KD-tree acceleration; visualize the curse of dimensionality;
and understand the lazy-vs-eager learner tradeoff.

**Prerequisites:** 01-01 (Python, NumPy & Tensor Speed)

**GPU Required:** No (classical ML)

## Part 0 — Setup & Prerequisites

k-Nearest Neighbors is the simplest possible supervised learning algorithm:
to classify a new point, find its k closest training examples and take a
majority vote. There is no explicit training phase — the model *is* the
training data. This "lazy learner" property creates a fundamental tradeoff:
zero training time but high inference time, which motivates spatial data
structures like the KD-tree for acceleration.

This notebook implements k-NN from scratch and reveals two deep insights:
(1) distance metrics profoundly affect which neighbors are "close", and
(2) in high dimensions all points become equidistant — the **curse of
dimensionality** — which limits k-NN's effectiveness and motivates the
dimensionality reduction techniques in Module 3.

In [ ]:
import sys
import time
import warnings
import math
from collections import Counter
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_digits, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

print(f"Python : {sys.version.split()[0]}")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")

In [ ]:
import random
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hyperparameters / constants ───────────────────────────────────────────────
K_DEFAULT      = 5          # default number of neighbours
K_MAX          = 31         # upper bound for k-sweep
TEST_SIZE      = 0.20       # 80/20 train/test split
COD_MAX_DIM    = 200        # max dimensions for curse-of-dimensionality plot
COD_N_SAMPLES  = 500        # samples for curse-of-dimensionality experiment
KD_N_QUERY     = 200        # how many test points to compare speed on

In [ ]:
# ── Digits dataset (8x8 handwritten digit images, 10 classes) ────────────────
digits = load_digits()
X_digits = digits.data.astype(np.float64)      # (1797, 64)
y_digits = digits.target                        # (1797,)

# 80/20 split — stratified so each class is proportionally represented
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_digits,
)

# Standard scaling: mean 0, std 1 per feature (important for distance metrics)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Digits  — train: {X_train.shape}  test: {X_test.shape}")
print(f"Classes : {sorted(np.unique(y_digits).tolist())}")
print(f"Class distribution (train): {dict(sorted(Counter(y_train).items()))}")

# ── make_blobs (for visual sanity checks) ─────────────────────────────────────
X_blobs, y_blobs = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=1.2,
    random_state=SEED,
)
X_blobs_tr, X_blobs_te, y_blobs_tr, y_blobs_te = train_test_split(
    X_blobs, y_blobs,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_blobs,
)
print(f"Blobs   — train: {X_blobs_tr.shape}  test: {X_blobs_te.shape}")

In [ ]:
# ── Visualise sample digits and blobs ─────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(14, 3))

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax = axes[0, digit]
    ax.imshow(X_train[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"'{digit}'", fontsize=9)
    ax.axis("off")

for digit in range(10):
    idx = np.where(y_train == digit)[0][1]
    ax = axes[1, digit]
    ax.imshow(X_train[idx].reshape(8, 8), cmap="gray_r")
    ax.axis("off")

plt.suptitle("Sample digits (scaled then reshaped for display)", fontsize=11)
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(5, 4))
colors = ["steelblue", "darkorange", "firebrick", "purple"]
for cls in range(4):
    mask = y_blobs_tr == cls
    ax2.scatter(X_blobs_tr[mask, 0], X_blobs_tr[mask, 1],
                color=colors[cls], label=f"Class {cls}", alpha=0.7, s=25)
ax2.set_title("make_blobs — 4 classes (train set)")
ax2.set_xlabel("Feature 0"); ax2.set_ylabel("Feature 1")
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## Part 1 — Distance Metrics & Core k-NN Logic

### Why Distance Metrics Matter

k-NN has no explicit model — it stores all training data and measures
**similarity** via a distance function at prediction time. The choice of
metric changes the geometry of the space and which neighbours are "nearest".

The three most common metrics in $\mathbb{R}^d$ are:

| Metric | Formula | Best for |
|--------|---------|----------|
| Euclidean ($L_2$) | $\sqrt{\sum_i (x_i - y_i)^2}$ | Continuous features, isotropic distributions |
| Manhattan ($L_1$) | $\sum_i |x_i - y_i|$ | Sparse or mixed-scale features, more robust to outliers |
| Cosine | $1 - \frac{x \cdot y}{\|x\|\,\|y\|}$ | Text, embeddings — direction matters more than magnitude |

All three are special cases of the **Minkowski metric**
$d_p(x, y) = \left(\sum_i |x_i - y_i|^p\right)^{1/p}$
with $p = 2$, $p = 1$, and the cosine variant respectively.

In [ ]:
def euclidean_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute the Euclidean (L2) distance between two 1-D vectors.

    Args:
        x1: First vector, shape (d,).
        x2: Second vector, shape (d,).

    Returns:
        Non-negative scalar L2 distance.
    """
    diff = x1 - x2
    return float(math.sqrt(np.dot(diff, diff)))


def manhattan_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute the Manhattan (L1) distance between two 1-D vectors.

    Args:
        x1: First vector, shape (d,).
        x2: Second vector, shape (d,).

    Returns:
        Non-negative scalar L1 distance.
    """
    return float(np.sum(np.abs(x1 - x2)))


def cosine_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute cosine distance (1 - cosine similarity) between two vectors.

    Cosine distance ranges from 0 (identical direction) to 2 (opposite
    direction). Undefined when either vector is the zero vector; returns 1.0
    in that degenerate case.

    Args:
        x1: First vector, shape (d,).
        x2: Second vector, shape (d,).

    Returns:
        Cosine distance in [0, 2].
    """
    norm1 = np.linalg.norm(x1)
    norm2 = np.linalg.norm(x2)
    if norm1 < 1e-12 or norm2 < 1e-12:
        return 1.0
    cos_sim = np.dot(x1, x2) / (norm1 * norm2)
    return float(1.0 - np.clip(cos_sim, -1.0, 1.0))


def pairwise_distances(
    X_query: np.ndarray,
    X_ref: np.ndarray,
    metric: str = "euclidean",
) -> np.ndarray:
    """Compute the full pairwise distance matrix between two sets of points.

    For each query point i and reference point j, computes D[i, j].
    Uses vectorised NumPy operations for speed (avoids Python-level loops
    over individual pairs).

    Args:
        X_query: Query points, shape (n_query, d).
        X_ref:   Reference points, shape (n_ref, d).
        metric:  One of "euclidean", "manhattan", "cosine".

    Returns:
        Distance matrix, shape (n_query, n_ref).

    Raises:
        ValueError: If metric is not one of the supported options.
    """
    if metric == "euclidean":
        # ||x - y||^2 = ||x||^2 + ||y||^2 - 2 x.y  (expanded)
        sq_norms_q = np.sum(X_query ** 2, axis=1, keepdims=True)  # (n_q, 1)
        sq_norms_r = np.sum(X_ref   ** 2, axis=1, keepdims=True)  # (n_r, 1)
        dot_products = X_query @ X_ref.T                            # (n_q, n_r)
        sq_dists = sq_norms_q + sq_norms_r.T - 2.0 * dot_products
        # Clip negatives caused by floating-point error before sqrt
        return np.sqrt(np.maximum(sq_dists, 0.0))

    elif metric == "manhattan":
        # Expand via broadcasting: (n_q, 1, d) - (1, n_r, d)
        diff = X_query[:, np.newaxis, :] - X_ref[np.newaxis, :, :]  # (n_q, n_r, d)
        return np.sum(np.abs(diff), axis=2)

    elif metric == "cosine":
        norms_q = np.linalg.norm(X_query, axis=1, keepdims=True) + 1e-12
        norms_r = np.linalg.norm(X_ref,   axis=1, keepdims=True) + 1e-12
        X_q_norm = X_query / norms_q
        X_r_norm = X_ref   / norms_r
        cos_sim = X_q_norm @ X_r_norm.T            # (n_q, n_r) in [-1, 1]
        return 1.0 - np.clip(cos_sim, -1.0, 1.0)

    else:
        raise ValueError(f"Unknown metric: {metric!r}. Choose 'euclidean', 'manhattan', or 'cosine'.")


# ── Quick unit tests ──────────────────────────────────────────────────────────
a = np.array([0.0, 0.0])
b = np.array([3.0, 4.0])
print(f"euclidean([0,0], [3,4]) = {euclidean_distance(a, b):.4f}  (expected 5.0)")
print(f"manhattan([0,0], [3,4]) = {manhattan_distance(a, b):.4f}  (expected 7.0)")

c = np.array([1.0, 0.0])
d_vec = np.array([0.0, 1.0])
print(f"cosine([1,0], [0,1])    = {cosine_distance(c, d_vec):.4f}  (expected 1.0 — perpendicular)")

e = np.array([1.0, 1.0])
print(f"cosine([1,0], [1,1])    = {cosine_distance(c, e):.4f}  (expected ~0.293)")

In [ ]:
# ── Visualise distance matrices for 30 digits from the test set ───────────────
N_VIS = 30
X_vis = X_test[:N_VIS]

D_eucl = pairwise_distances(X_vis, X_vis, metric="euclidean")
D_manh = pairwise_distances(X_vis, X_vis, metric="manhattan")
D_cos  = pairwise_distances(X_vis, X_vis, metric="cosine")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, D, title in zip(axes,
                         [D_eucl, D_manh, D_cos],
                         ["Euclidean", "Manhattan", "Cosine"]):
    im = ax.imshow(D, cmap="viridis")
    ax.set_title(f"{title} distance matrix\n(30 test samples)", fontsize=10)
    ax.set_xlabel("Sample index"); ax.set_ylabel("Sample index")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Pairwise Distance Matrices (Scaled Digits)", fontsize=12)
plt.tight_layout(); plt.show()

# Print scale comparison
print(f"Euclidean — mean: {D_eucl.mean():.3f},  max: {D_eucl.max():.3f}")
print(f"Manhattan — mean: {D_manh.mean():.3f},  max: {D_manh.max():.3f}")
print(f"Cosine    — mean: {D_cos.mean():.3f},   max: {D_cos.max():.3f}")

---
### The Curse of Dimensionality

As the number of dimensions $d$ grows, the ratio of the maximum to minimum
pairwise distance approaches 1:

$$\lim_{d \to \infty} \frac{\max_{x,y} \|x - y\|}{\min_{x,y} \|x - y\|} \to 1$$

In other words, all points become **equidistant** from each other.
When this happens, "nearest neighbour" is meaningless — every training point
is equally far from the query, so the vote is essentially random.

This is not merely theoretical: for uniformly distributed points in $[0,1]^d$,
the average distance grows as $\sqrt{d/3}$ while the *contrast* between
nearest and farthest neighbour shrinks. We can measure this empirically.

In [ ]:
def curse_of_dimensionality_experiment(
    n_samples: int,
    max_dim: int,
    dims: list[int] | None = None,
) -> tuple[list[int], list[float], list[float]]:
    """Measure how distance contrast degrades with dimensionality.

    For each dimensionality d, generates n_samples uniform random points in
    [0, 1]^d, computes all pairwise Euclidean distances, and records:
      - mean_distance: average pairwise distance
      - contrast: (max_dist - min_dist) / mean_dist  (lower = worse for k-NN)

    Args:
        n_samples: Number of random points to generate per dimension.
        max_dim:   Maximum dimensionality to test (used if dims is None).
        dims:      Explicit list of dimensions to test (overrides max_dim).

    Returns:
        Tuple (dim_list, mean_dists, contrasts).
            dim_list:    List of tested dimensionalities.
            mean_dists:  Mean pairwise distance at each dimensionality.
            contrasts:   (max - min) / mean contrast at each dimensionality.
    """
    if dims is None:
        # Sample log-spaced dimensions from 1 to max_dim
        dims = sorted(set(
            [1, 2, 3, 5] +
            list(np.unique(np.logspace(0, np.log10(max_dim), 25).astype(int)))
        ))

    rng = np.random.default_rng(SEED)
    dim_list: list[int] = []
    mean_dists: list[float] = []
    contrasts: list[float] = []

    for d in dims:
        X_rand = rng.uniform(0.0, 1.0, size=(n_samples, d))
        D = pairwise_distances(X_rand, X_rand, metric="euclidean")
        # Exclude self-distances (diagonal = 0)
        mask = ~np.eye(n_samples, dtype=bool)
        dists_flat = D[mask]

        d_min  = dists_flat.min()
        d_max  = dists_flat.max()
        d_mean = dists_flat.mean()

        contrast = (d_max - d_min) / (d_mean + 1e-12)

        dim_list.append(d)
        mean_dists.append(float(d_mean))
        contrasts.append(float(contrast))

    return dim_list, mean_dists, contrasts


dims_tested, avg_dists, contrasts = curse_of_dimensionality_experiment(
    n_samples=COD_N_SAMPLES,
    max_dim=COD_MAX_DIM,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(dims_tested, avg_dists, color="steelblue", linewidth=2, marker="o", markersize=3)
ax1.set_xlabel("Dimensionality d")
ax1.set_ylabel("Mean pairwise Euclidean distance")
ax1.set_title("Mean Distance Grows with Dimensionality")
ax1.set_xscale("log")
ax1.grid(True, alpha=0.3)

ax2.plot(dims_tested, contrasts, color="firebrick", linewidth=2, marker="o", markersize=3)
ax2.set_xlabel("Dimensionality d")
ax2.set_ylabel("Contrast  (max - min) / mean")
ax2.set_title("Distance Contrast Collapses\n(Curse of Dimensionality)")
ax2.set_xscale("log")
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")

plt.suptitle("Curse of Dimensionality: Uniform Points in [0,1]^d", fontsize=12)
plt.tight_layout(); plt.show()

print(f"d=  2: contrast = {contrasts[dims_tested.index(2)]:.3f}")
d5_idx = min(range(len(dims_tested)), key=lambda i: abs(dims_tested[i] - 5))
print(f"d=  5: contrast = {contrasts[d5_idx]:.3f}")
d50_idx = min(range(len(dims_tested)), key=lambda i: abs(dims_tested[i] - 50))
print(f"d= 50: contrast = {contrasts[d50_idx]:.3f}")
print(f"d={dims_tested[-1]:3d}: contrast = {contrasts[-1]:.3f}")
print("\nAs d grows, contrast → 0: all distances become nearly equal.")

---
### Weighted Voting

In **uniform k-NN**, each of the k neighbours casts one vote regardless of
how close it is. In **distance-weighted k-NN**, closer neighbours cast
stronger votes. If neighbour $i$ is at distance $d_i$, its weight is:

$$w_i = \frac{1}{d_i + \varepsilon}$$

where $\varepsilon$ is a small constant to avoid division by zero when a
training point coincides exactly with the query. The predicted class is:

$$\hat{y} = \arg\max_c \sum_{i=1}^k w_i \cdot \mathbf{1}[y_i = c]$$

Weighted voting is especially helpful when the k neighbours include a mix
of very close and more distant points.

---
## Part 2 — KNNClassifier: Putting It All Together

We assemble the distance functions and voting logic into a unified
`KNNClassifier` class with a standard sklearn-style API (`fit`, `predict`,
`score`). The class supports all three distance metrics and toggles between
uniform and distance-weighted voting.

In [ ]:
class KNNClassifier:
    """k-Nearest Neighbors classifier implemented from scratch.

    Implements brute-force k-NN: stores all training data and computes
    pairwise distances at prediction time. Supports Euclidean, Manhattan,
    and cosine distance metrics, and optionally weights votes by
    inverse distance.

    Attributes:
        k:        Number of neighbours to consider.
        metric:   Distance metric -- "euclidean", "manhattan", or "cosine".
        weighted: If True, weight votes by 1 / (distance + eps).
        X_train_: Stored training features, shape (n_train, d).
        y_train_: Stored training labels, shape (n_train,).
        classes_: Sorted unique class labels.
    """

    def __init__(
        self,
        k: int = 5,
        metric: str = "euclidean",
        weighted: bool = False,
    ) -> None:
        """Initialise KNNClassifier hyperparameters.

        Args:
            k:        Number of nearest neighbours (must be >= 1).
            metric:   Distance metric -- "euclidean", "manhattan", or "cosine".
            weighted: If True, weight votes by inverse distance.
        """
        self.k = k
        self.metric = metric
        self.weighted = weighted
        self.X_train_: np.ndarray | None = None
        self.y_train_: np.ndarray | None = None
        self.classes_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNNClassifier":
        """Store training data (k-NN has no explicit training step).

        Args:
            X: Training feature matrix, shape (n_train, d).
            y: Training labels, shape (n_train,).

        Returns:
            self (fitted estimator).
        """
        self.X_train_ = np.array(X, dtype=np.float64)
        self.y_train_ = np.array(y)
        self.classes_  = np.unique(y)
        return self

    def _vote(self, distances: np.ndarray, labels: np.ndarray) -> int:
        """Perform a k-NN vote (uniform or distance-weighted) for one query.

        Args:
            distances: Sorted distances to the k nearest neighbours, shape (k,).
            labels:    Corresponding labels for those k neighbours, shape (k,).

        Returns:
            Predicted class label (integer).
        """
        if self.weighted:
            weights = 1.0 / (distances + 1e-9)
            vote_totals: dict[int, float] = {}
            for w, lbl in zip(weights, labels):
                vote_totals[int(lbl)] = vote_totals.get(int(lbl), 0.0) + w
            return max(vote_totals, key=vote_totals.get)
        else:
            counts = Counter(int(lbl) for lbl in labels)
            return counts.most_common(1)[0][0]

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels for a query set.

        Args:
            X: Query feature matrix, shape (n_query, d).

        Returns:
            Predicted label array, shape (n_query,).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.X_train_ is None:
            raise RuntimeError("Call fit() before predict().")

        D = pairwise_distances(X, self.X_train_, metric=self.metric)
        # For each query, sort by distance and take top k
        predictions = np.empty(len(X), dtype=self.y_train_.dtype)
        for i in range(len(X)):
            sorted_idx = np.argsort(D[i])[:self.k]
            nn_dists  = D[i][sorted_idx]
            nn_labels = self.y_train_[sorted_idx]
            predictions[i] = self._vote(nn_dists, nn_labels)
        return predictions

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy on the given data.

        Args:
            X: Feature matrix, shape (n, d).
            y: True labels, shape (n,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))


# ── Sanity check on make_blobs (small, fast, visually verifiable) ─────────────
knn_blobs = KNNClassifier(k=5, metric="euclidean")
knn_blobs.fit(X_blobs_tr, y_blobs_tr)
acc_blobs = knn_blobs.score(X_blobs_te, y_blobs_te)
print(f"make_blobs test accuracy (k=5, Euclidean): {acc_blobs:.4f}")

# ── Decision boundary visualisation on make_blobs ────────────────────────────
h = 0.15
x_min, x_max = X_blobs[:, 0].min() - 1, X_blobs[:, 0].max() + 1
y_min, y_max = X_blobs[:, 1].min() - 1, X_blobs[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
grid_pts = np.c_[xx.ravel(), yy.ravel()]
Z = knn_blobs.predict(grid_pts).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.25, cmap="tab10")
for cls, col in zip(range(4), colors):
    mask = y_blobs_tr == cls
    ax.scatter(X_blobs_tr[mask, 0], X_blobs_tr[mask, 1],
               color=col, label=f"Class {cls}", s=30, edgecolors="k", linewidths=0.5)
te_preds = knn_blobs.predict(X_blobs_te)
ax.scatter(X_blobs_te[:, 0], X_blobs_te[:, 1],
           c=[colors[int(p)] for p in te_preds],
           marker="^", s=80, edgecolors="black", linewidths=1.0, label="Test (triangle)")
ax.set_title(f"k-NN Decision Boundary (k=5, Euclidean)\nacc={acc_blobs:.4f}")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
### KD-Tree Acceleration

Brute-force k-NN has $O(n \cdot d)$ query time — slow when $n$ is large.
The **KD-tree** (k-dimensional tree) recursively partitions the feature
space into axis-aligned hyper-rectangles, reducing average query time to
$O(\log n)$ in low dimensions (the benefit diminishes as $d$ grows — another
manifestation of the curse of dimensionality).

**Build phase:** At each node, choose the dimension with the highest spread,
split at the median, and recurse on the two halves.

**Query phase:** Traverse the tree to the leaf region containing the query
point. Backtrack and prune branches whose bounding box cannot contain a closer
neighbour than the current best — this pruning is what makes KD-trees fast.

We implement a 1-NN KD-tree here (the simplest useful case) to illustrate
the data structure. Production implementations (like sklearn's) extend this
to k-NN with priority queues.

In [ ]:
class KDNode:
    """A node in a KD-tree.

    Attributes:
        point:     The data point stored at this node, shape (d,).
        label:     Class label associated with this point.
        left:      Left child (points below split), or None if leaf.
        right:     Right child (points above split), or None if leaf.
        split_dim: Feature dimension used to split at this node.
        split_val: The split value (median of split_dim in the node's data).
    """

    def __init__(
        self,
        point: np.ndarray,
        label: int,
        left: "KDNode | None",
        right: "KDNode | None",
        split_dim: int,
        split_val: float,
    ) -> None:
        """Initialise a KD-tree node.

        Args:
            point:     Data point stored at this node.
            label:     Class label for this point.
            left:      Left child subtree.
            right:     Right child subtree.
            split_dim: Dimension index used to partition the data.
            split_val: Threshold value for the partition.
        """
        self.point     = point
        self.label     = label
        self.left      = left
        self.right     = right
        self.split_dim = split_dim
        self.split_val = split_val


class KDTree:
    """KD-tree for efficient 1-nearest-neighbour search.

    Builds a binary tree that recursively partitions the feature space
    along alternating axes. Each internal node stores one data point and
    splits on the axis with the maximum variance among remaining points.
    Supports only 1-NN queries (returns the single nearest neighbour's label).

    Attributes:
        root_:   Root KDNode of the built tree, or None for empty data.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        """Build the KD-tree from training data.

        Args:
            X: Training feature matrix, shape (n, d).
            y: Training class labels, shape (n,).
        """
        self.root_ = self._build(np.array(X, dtype=np.float64),
                                 np.array(y), depth=0)

    def _build(
        self,
        points: np.ndarray,
        labels: np.ndarray,
        depth: int,
    ) -> "KDNode | None":
        """Recursively build the KD-tree.

        Args:
            points: Data points for this subtree, shape (n, d).
            labels: Corresponding labels, shape (n,).
            depth:  Current tree depth (used to cycle through split dims).

        Returns:
            Root KDNode of the subtree, or None if points is empty.
        """
        if len(points) == 0:
            return None

        d = points.shape[1]
        # Split on the dimension with the largest spread (variance)
        split_dim = int(np.argmax(np.var(points, axis=0)))
        sorted_idx = np.argsort(points[:, split_dim])
        points  = points[sorted_idx]
        labels  = labels[sorted_idx]

        median_idx = len(points) // 2
        node = KDNode(
            point     = points[median_idx],
            label     = int(labels[median_idx]),
            left      = self._build(points[:median_idx], labels[:median_idx], depth + 1),
            right     = self._build(points[median_idx + 1:], labels[median_idx + 1:], depth + 1),
            split_dim = split_dim,
            split_val = float(points[median_idx, split_dim]),
        )
        return node

    def _search_1nn(
        self,
        node: "KDNode | None",
        query: np.ndarray,
        best: list,
    ) -> None:
        """Recursively search for the 1-nearest neighbour.

        Traverses the tree like a BST, then backtracks and prunes branches
        whose bounding distance exceeds the current best distance.

        Args:
            node:  Current node being examined (None = leaf sentinel).
            query: Query point, shape (d,).
            best:  Mutable list [best_dist, best_label] updated in place.
        """
        if node is None:
            return

        dist = float(np.sqrt(np.dot(query - node.point, query - node.point)))
        if dist < best[0]:
            best[0] = dist
            best[1] = node.label

        # Traverse toward the side containing the query point first
        diff = query[node.split_dim] - node.split_val
        near_branch  = node.left  if diff <= 0 else node.right
        far_branch   = node.right if diff <= 0 else node.left

        self._search_1nn(near_branch, query, best)

        # Prune: only explore far branch if the splitting hyperplane is closer
        # than the current best distance
        if abs(diff) < best[0]:
            self._search_1nn(far_branch, query, best)

    def query_1nn(self, X_query: np.ndarray) -> np.ndarray:
        """Return the 1-nearest-neighbour label for each query point.

        Args:
            X_query: Query feature matrix, shape (n_query, d).

        Returns:
            Predicted label array, shape (n_query,).
        """
        preds = np.empty(len(X_query), dtype=int)
        for i, q in enumerate(X_query):
            best = [float("inf"), -1]
            self._search_1nn(self.root_, q, best)
            preds[i] = best[1]
        return preds


# ── Sanity check: KD-tree 1-NN should match brute-force 1-NN ──────────────────
knn1_brute = KNNClassifier(k=1, metric="euclidean")
knn1_brute.fit(X_blobs_tr, y_blobs_tr)

kdtree = KDTree(X_blobs_tr, y_blobs_tr)

brute_preds = knn1_brute.predict(X_blobs_te)
kd_preds    = kdtree.query_1nn(X_blobs_te)

match_pct = np.mean(brute_preds == kd_preds)
print(f"KD-tree vs brute-force 1-NN agreement: {match_pct:.4f} (should be 1.0)")
print(f"Brute-force 1-NN test acc: {np.mean(brute_preds == y_blobs_te):.4f}")
print(f"KD-tree   1-NN test acc  : {np.mean(kd_preds == y_blobs_te):.4f}")

---
## Part 3 — Training & Application on Digits

We now apply the from-scratch `KNNClassifier` to the Digits dataset
(1797 samples, 64 features, 10 classes). We compare k values,
distance metrics, weighted vs uniform voting, and measure speed
differences between brute-force and KD-tree lookups.

In [ ]:
# ── Sweep k from 1 to K_MAX (odd values to avoid ties) ───────────────────────
k_values = list(range(1, K_MAX + 1, 2))
k_accs: list[float] = []

for k_val in k_values:
    clf = KNNClassifier(k=k_val, metric="euclidean", weighted=False)
    clf.fit(X_train, y_train)
    k_accs.append(clf.score(X_test, y_test))

best_k   = k_values[int(np.argmax(k_accs))]
best_acc = max(k_accs)

print(f"Best k = {best_k}  |  test accuracy = {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, k_accs, marker="o", markersize=5, linewidth=1.8,
        color="steelblue", label="Test accuracy")
ax.axvline(best_k, color="firebrick", linestyle="--", linewidth=1.5,
           label=f"Best k = {best_k}")
ax.set_xlabel("k (number of neighbours)")
ax.set_ylabel("Test accuracy")
ax.set_title("k-NN Accuracy vs k on Digits Dataset (Euclidean)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Note: k=1 often overfits (high variance); large k underfits (high bias)
print(f"k= 1: {k_accs[0]:.4f}  (high variance — memorises every point)")
print(f"k={best_k:2d}: {best_acc:.4f}  (best bias-variance tradeoff on this dataset)")
print(f"k={K_MAX:2d}: {k_accs[-1]:.4f}  (too smooth — high bias)")

In [ ]:
# ── Compare three distance metrics at the best k ─────────────────────────────
metrics = ["euclidean", "manhattan", "cosine"]
metric_results: dict[str, float] = {}

for metric in metrics:
    clf = KNNClassifier(k=best_k, metric=metric, weighted=False)
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    metric_results[metric] = acc
    print(f"  {metric:12s}  acc = {acc:.4f}")

# ── Weighted vs uniform voting ─────────────────────────────────────────────────
print()
for metric in metrics:
    clf_w = KNNClassifier(k=best_k, metric=metric, weighted=True)
    clf_w.fit(X_train, y_train)
    acc_w = clf_w.score(X_test, y_test)
    print(f"  {metric:12s} (weighted)  acc = {acc_w:.4f}")

# Bar chart comparison
labels_bar = metrics + [m + "\n(weighted)" for m in metrics]
values_bar = [metric_results[m] for m in metrics]
for metric in metrics:
    clf_w = KNNClassifier(k=best_k, metric=metric, weighted=True)
    clf_w.fit(X_train, y_train)
    values_bar.append(clf_w.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ["steelblue"] * 3 + ["darkorange"] * 3
bars = ax.bar(labels_bar, values_bar, color=bar_colors, edgecolor="black",
              linewidth=0.8, alpha=0.85)
for bar, val in zip(bars, values_bar):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylim(min(values_bar) - 0.02, 1.01)
ax.set_ylabel("Test accuracy")
ax.set_title(f"k-NN: Distance Metric Comparison (k={best_k}, Digits dataset)")
patch_u = mpatches.Patch(color="steelblue", label="Uniform voting")
patch_w = mpatches.Patch(color="darkorange", label="Weighted voting")
ax.legend(handles=[patch_u, patch_w]); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# ── Compare scratch implementation vs sklearn KNeighborsClassifier ───────────
clf_scratch = KNNClassifier(k=best_k, metric="euclidean", weighted=False)
clf_scratch.fit(X_train, y_train)

clf_sklearn = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean",
                                    algorithm="brute", weights="uniform")
clf_sklearn.fit(X_train, y_train)

acc_scratch = clf_scratch.score(X_test, y_test)
acc_sklearn = clf_sklearn.score(X_test, y_test)

print(f"From-scratch k-NN accuracy : {acc_scratch:.4f}")
print(f"sklearn      k-NN accuracy : {acc_sklearn:.4f}")
print(f"Match: {np.array_equal(clf_scratch.predict(X_test), clf_sklearn.predict(X_test))}")

# ── Speed comparison: brute-force scratch vs sklearn brute vs sklearn KD-tree ─
t0 = time.perf_counter()
_ = clf_scratch.predict(X_test)
t_scratch = time.perf_counter() - t0

clf_sk_kd = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean",
                                   algorithm="kd_tree", weights="uniform")
clf_sk_kd.fit(X_train, y_train)

t0 = time.perf_counter()
_ = clf_sklearn.predict(X_test)
t_sk_brute = time.perf_counter() - t0

t0 = time.perf_counter()
_ = clf_sk_kd.predict(X_test)
t_sk_kd = time.perf_counter() - t0

print(f"\nPrediction time on {len(X_test)} test samples:")
print(f"  Scratch brute-force : {t_scratch*1000:.1f} ms")
print(f"  sklearn brute-force : {t_sk_brute*1000:.1f} ms")
print(f"  sklearn KD-tree     : {t_sk_kd*1000:.1f} ms")
print(f"  Scratch / sklearn brute ratio: {t_scratch/t_sk_brute:.1f}x")

In [ ]:
# ── Our KD-tree vs brute-force scratch on Digits (1-NN only) ─────────────────
# Subsample query points to keep the demo fast
rng_speed = np.random.default_rng(SEED)
query_idx = rng_speed.choice(len(X_test), size=min(KD_N_QUERY, len(X_test)),
                              replace=False)
X_query_sub = X_test[query_idx]
y_query_sub = y_test[query_idx]

# Build KD-tree from training data
t0 = time.perf_counter()
kdtree_digits = KDTree(X_train, y_train)
t_build = time.perf_counter() - t0

# KD-tree query
t0 = time.perf_counter()
kd_preds_digits = kdtree_digits.query_1nn(X_query_sub)
t_kd = time.perf_counter() - t0

# Brute-force 1-NN query
knn1_scratch = KNNClassifier(k=1, metric="euclidean")
knn1_scratch.fit(X_train, y_train)
t0 = time.perf_counter()
bf_preds_digits = knn1_scratch.predict(X_query_sub)
t_bf = time.perf_counter() - t0

kd_acc = float(np.mean(kd_preds_digits == y_query_sub))
bf_acc = float(np.mean(bf_preds_digits == y_query_sub))
agreement = float(np.mean(kd_preds_digits == bf_preds_digits))

print(f"KD-tree build time : {t_build*1000:.1f} ms")
print(f"\nQuery on {len(X_query_sub)} test samples (1-NN):")
print(f"  Scratch brute-force: {t_bf*1000:.2f} ms  |  acc = {bf_acc:.4f}")
print(f"  Scratch KD-tree    : {t_kd*1000:.2f} ms  |  acc = {kd_acc:.4f}")
print(f"  Agreement (same pred): {agreement:.4f}")
print()
print("Note: For 64-D data the KD-tree speedup is modest (curse of dimensionality).")
print("KD-trees shine in 2-D or 3-D spatial problems; brute-force scales better in high-D.")

---
### k-NN Regression

k-NN extends naturally to regression: instead of a majority vote, predict
the **mean** (or distance-weighted mean) of the k nearest neighbours'
target values. Weighted regression is especially useful when neighbours
vary widely in distance — closer points should exert more influence.

$$\hat{y} = \frac{\sum_{i=1}^k w_i y_i}{\sum_{i=1}^k w_i}, \quad
w_i = \frac{1}{d_i + \varepsilon}$$

In [ ]:
class KNNRegressor:
    """k-Nearest Neighbors regressor implemented from scratch.

    Predicts continuous target values by averaging (or distance-weighted
    averaging) the k nearest training targets. Uses the same vectorised
    pairwise distance computation as KNNClassifier.

    Attributes:
        k:        Number of neighbours to consider.
        metric:   Distance metric -- "euclidean", "manhattan", or "cosine".
        weighted: If True, weight predictions by 1 / (distance + eps).
        X_train_: Stored training features, shape (n_train, d).
        y_train_: Stored training targets, shape (n_train,).
    """

    def __init__(
        self,
        k: int = 5,
        metric: str = "euclidean",
        weighted: bool = False,
    ) -> None:
        """Initialise KNNRegressor hyperparameters.

        Args:
            k:        Number of nearest neighbours (must be >= 1).
            metric:   Distance metric -- "euclidean", "manhattan", or "cosine".
            weighted: If True, weight predictions by inverse distance.
        """
        self.k = k
        self.metric = metric
        self.weighted = weighted
        self.X_train_: np.ndarray | None = None
        self.y_train_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNNRegressor":
        """Store training data (k-NN has no explicit training step).

        Args:
            X: Training feature matrix, shape (n_train, d).
            y: Training target values, shape (n_train,).

        Returns:
            self (fitted estimator).
        """
        self.X_train_ = np.array(X, dtype=np.float64)
        self.y_train_ = np.array(y, dtype=np.float64)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict continuous target values for a query set.

        Args:
            X: Query feature matrix, shape (n_query, d).

        Returns:
            Predicted value array, shape (n_query,).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.X_train_ is None:
            raise RuntimeError("Call fit() before predict().")

        D = pairwise_distances(X, self.X_train_, metric=self.metric)
        preds = np.empty(len(X), dtype=np.float64)

        for i in range(len(X)):
            sorted_idx = np.argsort(D[i])[:self.k]
            nn_dists   = D[i][sorted_idx]
            nn_targets = self.y_train_[sorted_idx]

            if self.weighted:
                weights = 1.0 / (nn_dists + 1e-9)
                preds[i] = float(np.dot(weights, nn_targets) / weights.sum())
            else:
                preds[i] = float(nn_targets.mean())

        return preds

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return coefficient of determination R^2 on the given data.

        R^2 = 1 - SS_res / SS_tot, where SS_res = sum((y - yhat)^2)
        and SS_tot = sum((y - mean(y))^2). Returns 0 when yhat = mean(y).

        Args:
            X: Feature matrix, shape (n, d).
            y: True target values, shape (n,).

        Returns:
            R-squared score in (-inf, 1].
        """
        y_pred = self.predict(X)
        ss_res = float(np.sum((y - y_pred) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        if ss_tot < 1e-12:
            return 1.0 if ss_res < 1e-12 else 0.0
        return float(1.0 - ss_res / ss_tot)


# ── Demo: k-NN regression on a synthetic 1-D sinusoidal signal ────────────────
rng_reg = np.random.default_rng(SEED)
n_reg = 200
X_reg_raw = rng_reg.uniform(0, 2 * np.pi, size=(n_reg, 1))
y_reg = np.sin(X_reg_raw[:, 0]) + rng_reg.normal(0, 0.2, size=n_reg)

X_reg_tr_raw, X_reg_te_raw, y_reg_tr, y_reg_te = train_test_split(
    X_reg_raw, y_reg, test_size=TEST_SIZE, random_state=SEED
)
# Standardise
mu_reg = X_reg_tr_raw.mean(axis=0); std_reg = X_reg_tr_raw.std(axis=0) + 1e-8
X_reg_tr = (X_reg_tr_raw - mu_reg) / std_reg
X_reg_te = (X_reg_te_raw - mu_reg) / std_reg

k_reg_vals = [1, 3, 7, 15, 30]
fig_reg, axes_reg = plt.subplots(1, len(k_reg_vals), figsize=(16, 3), sharey=True)

X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)
X_plot_std = (X_plot - mu_reg) / std_reg

for ax_r, k_r in zip(axes_reg, k_reg_vals):
    reg_u = KNNRegressor(k=k_r, metric="euclidean", weighted=False)
    reg_u.fit(X_reg_tr, y_reg_tr)
    reg_w = KNNRegressor(k=k_r, metric="euclidean", weighted=True)
    reg_w.fit(X_reg_tr, y_reg_tr)

    y_plot_u = reg_u.predict(X_plot_std)
    y_plot_w = reg_w.predict(X_plot_std)
    r2_u = reg_u.score(X_reg_te, y_reg_te)
    r2_w = reg_w.score(X_reg_te, y_reg_te)

    ax_r.scatter(X_reg_raw[:, 0], y_reg, s=8, color="gray", alpha=0.4, label="Data")
    ax_r.plot(X_plot[:, 0], np.sin(X_plot[:, 0]), "k--", linewidth=1.5, label="True sin")
    ax_r.plot(X_plot[:, 0], y_plot_u, color="steelblue",  linewidth=1.8,
              label=f"Uniform R2={r2_u:.3f}")
    ax_r.plot(X_plot[:, 0], y_plot_w, color="darkorange", linewidth=1.8, linestyle="--",
              label=f"Weighted R2={r2_w:.3f}")
    ax_r.set_title(f"k={k_r}", fontsize=10)
    ax_r.set_xlabel("x"); ax_r.legend(fontsize=6)

axes_reg[0].set_ylabel("y = sin(x) + noise")
plt.suptitle("k-NN Regression: Effect of k and Weighting on sin(x) + noise", fontsize=11)
plt.tight_layout(); plt.show()

# Print R2 summary
print("k-NN Regression R2 on test set (sin signal):")
for k_r in k_reg_vals:
    reg = KNNRegressor(k=k_r, metric="euclidean", weighted=True)
    reg.fit(X_reg_tr, y_reg_tr)
    print(f"  k={k_r:2d} (weighted): R2 = {reg.score(X_reg_te, y_reg_te):.4f}")

---
### Choosing k with Cross-Validation

Selecting k via a single hold-out split can overfit to that particular split.
**K-fold cross-validation** is more reliable: partition the training set into
K equal folds, train on K-1 folds and evaluate on the remaining fold, and
average the K scores. The k that maximises average CV accuracy is selected.

In [ ]:
def kfold_indices(
    n: int,
    n_folds: int,
    seed: int = SEED,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Generate train/val index splits for stratification-free K-fold CV.

    Shuffles indices then partitions them into n_folds consecutive blocks,
    returning (train_idx, val_idx) for each fold.

    Args:
        n:       Total number of samples.
        n_folds: Number of folds (K in K-fold CV).
        seed:    Random seed for shuffling.

    Returns:
        List of (train_idx, val_idx) tuples, length n_folds.
    """
    rng_cv = np.random.default_rng(seed)
    indices = rng_cv.permutation(n)
    fold_sizes = [n // n_folds + (1 if i < n % n_folds else 0) for i in range(n_folds)]
    splits: list[tuple[np.ndarray, np.ndarray]] = []
    start = 0
    for fold_size in fold_sizes:
        val_idx   = indices[start : start + fold_size]
        train_idx = np.concatenate([indices[:start], indices[start + fold_size:]])
        splits.append((train_idx, val_idx))
        start += fold_size
    return splits


def cross_val_knn(
    X: np.ndarray,
    y: np.ndarray,
    k_values: list[int],
    n_folds: int = 5,
    metric: str = "euclidean",
) -> dict[int, float]:
    """Evaluate k-NN accuracy for multiple k via K-fold cross-validation.

    For each k in k_values, runs n_folds CV folds on the training data and
    returns the mean accuracy over folds. This provides a more reliable
    estimate of generalisation performance than a single hold-out split.

    Args:
        X:        Feature matrix, shape (n, d).
        y:        Label array, shape (n,).
        k_values: List of k values to evaluate.
        n_folds:  Number of CV folds.
        metric:   Distance metric for k-NN.

    Returns:
        Dictionary mapping each k to its mean CV accuracy.
    """
    splits = kfold_indices(len(X), n_folds)
    results: dict[int, float] = {}

    for k_val in k_values:
        fold_accs: list[float] = []
        for train_idx, val_idx in splits:
            X_tr_fold, y_tr_fold = X[train_idx], y[train_idx]
            X_va_fold, y_va_fold = X[val_idx],   y[val_idx]

            clf_fold = KNNClassifier(k=k_val, metric=metric)
            clf_fold.fit(X_tr_fold, y_tr_fold)
            fold_accs.append(clf_fold.score(X_va_fold, y_va_fold))

        results[k_val] = float(np.mean(fold_accs))

    return results


# Run 5-fold CV on Digits training set (odd k values up to 21)
cv_k_values = list(range(1, 22, 2))
print("Running 5-fold CV on Digits training set...")
cv_results = cross_val_knn(X_train, y_train, cv_k_values, n_folds=5)

best_cv_k  = max(cv_results, key=cv_results.get)
best_cv_acc = cv_results[best_cv_k]
print(f"Best k (CV) = {best_cv_k}  |  mean CV accuracy = {best_cv_acc:.4f}")

# Test accuracy with CV-selected k
clf_cv_best = KNNClassifier(k=best_cv_k, metric="euclidean")
clf_cv_best.fit(X_train, y_train)
test_acc_cv_k = clf_cv_best.score(X_test, y_test)
print(f"Test accuracy with k={best_cv_k} (CV-selected): {test_acc_cv_k:.4f}")
print(f"Test accuracy with k={best_k}   (hold-out selected): {best_acc:.4f}")

# Plot CV vs hold-out k selection
fig_cv, ax_cv = plt.subplots(figsize=(9, 4))
cv_accs  = [cv_results[k_val] for k_val in cv_k_values]
ho_accs  = [KNNClassifier(k=k_val).fit(X_train, y_train).score(X_test, y_test)
            for k_val in cv_k_values]

ax_cv.plot(cv_k_values, cv_accs, marker="o", markersize=5, linewidth=1.8,
           color="darkorange", label="5-fold CV accuracy")
ax_cv.plot(cv_k_values, ho_accs, marker="s", markersize=5, linewidth=1.8,
           color="steelblue", linestyle="--", label="Hold-out test accuracy")
ax_cv.axvline(best_cv_k, color="darkorange", linestyle=":", label=f"CV best k={best_cv_k}")
ax_cv.axvline(best_k,    color="steelblue",  linestyle=":", label=f"HO best k={best_k}")
ax_cv.set_xlabel("k (number of neighbours)")
ax_cv.set_ylabel("Accuracy")
ax_cv.set_title("k-NN: 5-Fold CV vs Hold-Out k Selection (Digits)")
ax_cv.legend(fontsize=9); ax_cv.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
### Radius-Based Nearest Neighbours

A natural variant of k-NN: instead of fixing the number of neighbours k,
fix a radius $r$ and let **all training points within distance $r$** vote.
This adapts the neighbourhood size to local data density — dense regions
get many votes from many neighbours; sparse regions get few.

The downside: if no training point falls within radius $r$ of a query,
prediction requires a fallback strategy (e.g., the overall majority class).

In [ ]:
class RadiusNeighborsClassifier:
    """Radius-based nearest neighbours classifier implemented from scratch.

    Unlike k-NN, which uses a fixed count k, this classifier considers ALL
    training points within a given radius r. The predicted class is the
    majority vote (or distance-weighted vote) among those neighbours.
    Falls back to the overall training majority class when no neighbours
    fall within the radius.

    Attributes:
        radius:         Radius threshold for neighbourhood definition.
        metric:         Distance metric -- "euclidean", "manhattan", "cosine".
        weighted:       If True, weight votes by inverse distance.
        X_train_:       Stored training features, shape (n_train, d).
        y_train_:       Stored training labels, shape (n_train,).
        fallback_class_: Majority class used when no neighbours are in radius.
    """

    def __init__(
        self,
        radius: float = 1.0,
        metric: str = "euclidean",
        weighted: bool = False,
    ) -> None:
        """Initialise RadiusNeighborsClassifier.

        Args:
            radius:   Neighbourhood radius (same units as the distance metric).
            metric:   Distance metric -- "euclidean", "manhattan", or "cosine".
            weighted: If True, weight votes by 1 / (distance + eps).
        """
        self.radius   = radius
        self.metric   = metric
        self.weighted = weighted
        self.X_train_: np.ndarray | None = None
        self.y_train_: np.ndarray | None = None
        self.fallback_class_: int = 0

    def fit(self, X: np.ndarray, y: np.ndarray) -> "RadiusNeighborsClassifier":
        """Store training data and compute the fallback majority class.

        Args:
            X: Training feature matrix, shape (n_train, d).
            y: Training labels, shape (n_train,).

        Returns:
            self (fitted estimator).
        """
        self.X_train_ = np.array(X, dtype=np.float64)
        self.y_train_ = np.array(y)
        counts = Counter(int(lbl) for lbl in y)
        self.fallback_class_ = counts.most_common(1)[0][0]
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels using all neighbours within the radius.

        Args:
            X: Query feature matrix, shape (n_query, d).

        Returns:
            Predicted label array, shape (n_query,).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.X_train_ is None:
            raise RuntimeError("Call fit() before predict().")

        D = pairwise_distances(X, self.X_train_, metric=self.metric)
        predictions = np.empty(len(X), dtype=self.y_train_.dtype)

        for i in range(len(X)):
            in_radius = np.where(D[i] <= self.radius)[0]

            if len(in_radius) == 0:
                predictions[i] = self.fallback_class_
                continue

            nn_labels = self.y_train_[in_radius]
            nn_dists  = D[i][in_radius]

            if self.weighted:
                weights = 1.0 / (nn_dists + 1e-9)
                vote_totals: dict[int, float] = {}
                for w, lbl in zip(weights, nn_labels):
                    key = int(lbl)
                    vote_totals[key] = vote_totals.get(key, 0.0) + w
                predictions[i] = max(vote_totals, key=vote_totals.get)
            else:
                counts_r = Counter(int(lbl) for lbl in nn_labels)
                predictions[i] = counts_r.most_common(1)[0][0]

        return predictions

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy on the given data.

        Args:
            X: Feature matrix, shape (n, d).
            y: True labels, shape (n,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))


# ── Compare radius-NN vs k-NN on Digits ───────────────────────────────────────
# The Euclidean distance between standardised 64-D digit features
# has a mean around 7-12; we sweep radius around that range
radius_vals = [3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]
print("Radius-NN vs k-NN on Digits (Euclidean):")
print(f"  {'Model':<28} {'Acc':>8}")
print(f"  {'-'*38}")

for r_val in radius_vals:
    rnn = RadiusNeighborsClassifier(radius=r_val, metric="euclidean")
    rnn.fit(X_train, y_train)
    acc_r = rnn.score(X_test, y_test)
    print(f"  Radius-NN (r={r_val:.1f})           {acc_r:.4f}")

print()
for k_val in [3, 5, best_k]:
    clf_k = KNNClassifier(k=k_val, metric="euclidean")
    clf_k.fit(X_train, y_train)
    acc_k = clf_k.score(X_test, y_test)
    print(f"  k-NN (k={k_val:2d})                  {acc_k:.4f}")

---
## Part 4 — Evaluation & Analysis

We evaluate the best k-NN configuration via confusion matrix, identify
which digit pairs are most confused, examine individual error cases, and
summarise the key algorithmic tradeoffs in a lazy-vs-eager comparison table.

In [ ]:
# ── Best model: euclidean, best_k, uniform voting ─────────────────────────────
clf_best = KNNClassifier(k=best_k, metric="euclidean", weighted=False)
clf_best.fit(X_train, y_train)
y_pred = clf_best.predict(X_test)

final_acc = float(np.mean(y_pred == y_test))
print(f"Final test accuracy: {final_acc:.4f}")

# Per-class accuracy
for digit in range(10):
    mask = y_test == digit
    per_acc = float(np.mean(y_pred[mask] == y_test[mask]))
    print(f"  Digit {digit}: {per_acc:.4f}  ({mask.sum()} samples)")

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format="d")
ax.set_title(f"Confusion Matrix — k-NN (k={best_k}, Euclidean)\nDigits Test Set, acc={final_acc:.4f}")
plt.tight_layout(); plt.show()

# Most confused pairs
off_diag = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
off_diag.sort(reverse=True)
print("\nTop-5 most confused digit pairs (true → predicted):")
for count, true_cls, pred_cls in off_diag[:5]:
    print(f"  {true_cls} → {pred_cls}: {count} errors")

In [ ]:
# ── Visualise misclassified digits ────────────────────────────────────────────
errors_idx = np.where(y_pred != y_test)[0]
print(f"Total errors: {len(errors_idx)} / {len(y_test)}")

# Show up to 20 misclassified examples
n_show = min(20, len(errors_idx))
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for plot_i, err_i in enumerate(errors_idx[:n_show]):
    ax = axes[plot_i // 10, plot_i % 10]
    # Reconstruct pixel image from scaled features
    img = X_test[err_i].reshape(8, 8)
    ax.imshow(img, cmap="gray_r")
    ax.set_title(f"T:{y_test[err_i]} P:{y_pred[err_i]}", fontsize=7)
    ax.axis("off")
plt.suptitle(f"Misclassified Digits (T=true, P=predicted) — {len(errors_idx)} total errors",
             fontsize=10)
plt.tight_layout(); plt.show()

# ── Ablation: effect of k on error count ──────────────────────────────────────
print("\nError count by k (odd k, Euclidean, uniform):")
for k_val in [1, 3, best_k, 11, 21, K_MAX]:
    clf_tmp = KNNClassifier(k=k_val, metric="euclidean")
    clf_tmp.fit(X_train, y_train)
    n_err = int(np.sum(clf_tmp.predict(X_test) != y_test))
    print(f"  k={k_val:2d}: {n_err:3d} errors  ({n_err/len(y_test):.2%} error rate)")

In [ ]:
# ── Lazy-vs-Eager Learner Comparison Table ────────────────────────────────────
# Compare k-NN (lazy) against Decision Tree and Logistic Regression (eager)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

models: dict[str, object] = {
    "k-NN (k=5, Euclid)": KNeighborsClassifier(n_neighbors=5, metric="euclidean"),
    "k-NN (k=best)":      KNeighborsClassifier(n_neighbors=best_k, metric="euclidean"),
    "Decision Tree":       DecisionTreeClassifier(max_depth=10, random_state=SEED),
    "Logistic Reg. (L2)":  LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
}

print(f"{'Model':<25} {'Train Time':>12} {'Pred Time':>10} {'Test Acc':>10}")
print("-" * 62)
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t_train = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = model.predict(X_test)
    t_pred = time.perf_counter() - t0

    acc = float(np.mean(preds == y_test))
    print(f"{name:<25} {t_train*1000:>10.1f}ms {t_pred*1000:>9.1f}ms {acc:>10.4f}")

print()
print("Key insight:")
print("  k-NN (lazy learner)   — near-zero train time, prediction scales with n_train")
print("  Decision Tree (eager) — non-trivial train time, fast O(depth) prediction")
print("  Logistic Reg (eager)  — train time dominated by optimisation, trivial prediction")

---
### Permutation Feature Importance for k-NN

k-NN has no built-in feature importance (unlike Decision Trees or Linear Models).
We estimate it via **permutation importance**: randomly shuffle one feature column,
measure how much accuracy drops, and repeat. A large drop means the feature was
important; no drop means k-NN barely used it.

Permutation importance is model-agnostic and works for any sklearn-compatible
estimator, making it a universal feature attribution technique.

In [ ]:
def permutation_feature_importance(
    clf: "KNNClassifier",
    X: np.ndarray,
    y: np.ndarray,
    n_repeats: int = 5,
    seed: int = SEED,
) -> np.ndarray:
    """Compute permutation feature importance for a fitted classifier.

    For each feature column, randomly shuffles its values n_repeats times and
    measures the mean accuracy drop vs baseline. A larger drop = more important
    feature. This is model-agnostic and requires only that clf.score() works.

    Args:
        clf:       Fitted classifier with a .score(X, y) method.
        X:         Feature matrix to evaluate on, shape (n, d).
        y:         True labels, shape (n,).
        n_repeats: Number of shuffle repetitions per feature (reduces variance).
        seed:      Random seed for reproducible shuffles.

    Returns:
        Importance array, shape (d,), where importance[j] = mean accuracy drop
        when feature j is permuted. Higher = more important.
    """
    rng_imp = np.random.default_rng(seed)
    baseline_acc = clf.score(X, y)
    n_features = X.shape[1]
    importances = np.zeros(n_features)

    for j in range(n_features):
        drops: list[float] = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[:, j] = rng_imp.permutation(X_perm[:, j])
            perm_acc = clf.score(X_perm, y)
            drops.append(baseline_acc - perm_acc)
        importances[j] = float(np.mean(drops))

    return importances


# Compute permutation importance for the best k-NN on Digits test set
# Use a smaller n_repeats for speed (64 features x 5 repeats = 320 evaluations)
print("Computing permutation feature importance (64 features x 3 repeats)...")
clf_for_imp = KNNClassifier(k=best_k, metric="euclidean")
clf_for_imp.fit(X_train, y_train)
importances = permutation_feature_importance(clf_for_imp, X_test, y_test,
                                              n_repeats=3, seed=SEED)

# Visualise as an 8x8 importance heatmap (matches the Digits pixel layout)
imp_image = importances.reshape(8, 8)
fig_imp, (ax_imp1, ax_imp2) = plt.subplots(1, 2, figsize=(11, 4))

im = ax_imp1.imshow(imp_image, cmap="hot")
ax_imp1.set_title(f"Permutation Importance (k={best_k}, Euclidean)\n"
                  "Pixel layout: 8x8 digit grid")
ax_imp1.set_xlabel("Pixel column"); ax_imp1.set_ylabel("Pixel row")
plt.colorbar(im, ax=ax_imp1)

# Bar chart of top-15 most important features
top_k_feat = np.argsort(importances)[::-1][:15]
ax_imp2.barh(range(15), importances[top_k_feat], color="steelblue", alpha=0.8)
ax_imp2.set_yticks(range(15))
ax_imp2.set_yticklabels([f"Pixel ({t//8},{t%8})" for t in top_k_feat], fontsize=8)
ax_imp2.invert_yaxis()
ax_imp2.set_xlabel("Mean accuracy drop when shuffled")
ax_imp2.set_title("Top-15 Most Important Pixels")
ax_imp2.grid(True, alpha=0.3, axis="x")

plt.suptitle("Permutation Feature Importance: k-NN on Digits", fontsize=12)
plt.tight_layout(); plt.show()

print(f"Baseline test accuracy: {clf_for_imp.score(X_test, y_test):.4f}")
print(f"Most important pixel:   row={top_k_feat[0]//8}, col={top_k_feat[0]%8}")
print(f"Least important pixels have importance ~{importances.min():.4f}")
print("Central pixels (rows 2-5, cols 2-5) tend to matter most — they carry")
print("the digit's core shape, while border pixels are often empty/background.")

# ── PCA preprocessing: reduce 64-D Digits to lower-D, then apply k-NN ────────
from sklearn.decomposition import PCA

pca_dims = [2, 5, 10, 20, 30, 40, 64]
pca_accs: list[float] = []

for n_comp in pca_dims:
    pca = PCA(n_components=n_comp, random_state=SEED)
    X_tr_pca = pca.fit_transform(X_train)
    X_te_pca = pca.transform(X_test)
    clf_pca = KNNClassifier(k=best_k, metric="euclidean")
    clf_pca.fit(X_tr_pca, y_tr_test := y_train)
    pca_accs.append(clf_pca.score(X_te_pca, y_test))

print()
print("k-NN accuracy after PCA dimensionality reduction (Digits, 64 -> d dims):")
for d, acc in zip(pca_dims, pca_accs):
    var_exp = PCA(n_components=d, random_state=SEED).fit(X_train).explained_variance_ratio_.sum()
    print(f"  d={d:3d}  acc={acc:.4f}  var_explained={var_exp:.2%}")

fig_pca, ax_pca = plt.subplots(figsize=(8, 4))
ax_pca.plot(pca_dims, pca_accs, marker="o", linewidth=2, color="steelblue",
            markersize=7, label="k-NN after PCA")
ax_pca.axhline(best_acc, color="firebrick", linestyle="--", linewidth=1.5,
               label=f"k-NN (no PCA) = {best_acc:.4f}")
ax_pca.set_xlabel("PCA components retained")
ax_pca.set_ylabel("Test accuracy")
ax_pca.set_title(f"Effect of PCA Preprocessing on k-NN Accuracy (k={best_k}, Digits)")
ax_pca.legend(); ax_pca.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Summary table: PCA preprocessing effect ───────────────────────────────────
print()
print("=" * 55)
print("Summary: PCA Preprocessing vs Direct k-NN")
print("=" * 55)
print(f"{'PCA dims':>10}  {'Accuracy':>10}  {'vs Full':>8}  {'Speedup':>8}")
print("-" * 55)

for n_comp, acc in zip(pca_dims, pca_accs):
    delta = acc - best_acc
    # Speedup: full 64-D distance is 64 mults/adds; n_comp-D is fewer
    speedup = 64.0 / n_comp if n_comp < 64 else 1.0
    print(f"{n_comp:>10d}  {acc:>10.4f}  {delta:>+8.4f}  {speedup:>7.1f}x")

print("-" * 55)
print(f"{'Full (64)':>10}  {best_acc:>10.4f}  {'baseline':>8}")
print()
print("Insight: PCA with 20-30 components retains nearly full accuracy")
print("while reducing the distance computation cost by 2-3x, which is")
print("especially valuable for brute-force k-NN on large datasets.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **k-NN is a lazy learner**: training is free (store data), but prediction
  costs $O(n \cdot d)$ per query in the brute-force case. This tradeoff is
  fundamental and opposite to most parametric models.
- **Distance metric matters**: Euclidean, Manhattan, and cosine distances
  produce different neighbourhood geometries. Feature scaling (standardisation)
  is essential before applying any distance-based method.
- **Curse of dimensionality**: in high-dimensional spaces all pairwise
  distances converge, making "nearest neighbour" nearly meaningless. This
  motivates dimensionality reduction (Module 3) before applying k-NN.
- **Choosing k**: small k → high variance (memorisation); large k → high bias
  (over-smoothing). Cross-validation on a held-out set selects the optimal k.
- **KD-trees** accelerate k-NN to $O(\log n)$ per query in low dimensions
  (2–20 features) but lose their advantage in high dimensions — brute-force
  with vectorised NumPy often wins above ~30 features.

### What's Next
→ **02-08 (Naive Bayes for Text Classification)** applies probabilistic
reasoning — Bayes' theorem and conditional independence — to classify text,
introducing bag-of-words and TF-IDF representations as precursors to
the word embedding approaches in Module 7.